# regression_health — exploratory analysis

Flaredown Autoimmune Symptom Tracker → predicting **next-day symptom severity (0–4)**.

Source: [Kaggle](https://www.kaggle.com/datasets/flaredown/flaredown-autoimmune-symptom-tracker?resource=download) · see `references/data_source.md`.

This notebook is the *interactive* companion to the scripted pipeline in `src/`.
It reads the already-built daily panel so it stays fast; rebuild it with
`python src/build_panel.py` if needed.

**Leakage rule (ToU 'Common Mistakes to Avoid'):** any statistic used for modeling
is computed on the training split only — never on the full dataset or the test period.

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PANEL = ROOT / 'data' / 'interim' / 'daily_panel.csv.gz'
pd.set_option('display.max_columns', 50)
print('panel exists:', PANEL.exists())

## 1. Raw profile (machine-readable)
Full-file profile produced by `src/profile_data.py`.

In [ ]:
prof = json.load(open(ROOT / 'reports' / 'profile_result.json'))
print('rows  :', f"{prof['total_rows']:,}")
print('users :', f"{prof['n_users']:,}")
print('dates :', prof['date_min'], '->', prof['date_max'])
print('\nnull rate (%):'); print(pd.Series(prof['null_rate']))
print('\nrows by trackable_type:'); print(pd.Series(prof['type_counts']))

## 2. The target — symptom severity distribution
`Symptom` rows carry an ordinal 0–4 severity. Right-skewed toward low values,
which is why a naive 'tomorrow = today' baseline is already fairly strong.

In [ ]:
h = prof['symptom_value_hist']
plt.bar(list(h.keys()), list(h.values()), color='#2563eb')
plt.title('Symptom severity distribution (target source)')
plt.xlabel('trackable_value'); plt.ylabel('rows'); plt.show()
print('mean', prof['symptom_value_pct']['mean'], '| median', prof['symptom_value_pct']['median'])

## 3. Daily panel — one row per (user, day)
Built by `src/build_panel.py`: long event log → daily features.

In [ ]:
p = pd.read_csv(PANEL, parse_dates=['date'])
print(p.shape, '| users:', p['user_id'].nunique())
p.head()

### Data-quality check: `age` (ToU 'Identifying Data Quality Issues')
Raw age contains impossible values (negatives, birth years). Inspect, then clip.

In [ ]:
print('raw age range:', p['age'].min(), '->', p['age'].max())
print('rows with age outside [5,120]:', int(((p['age'] < 5) | (p['age'] > 120)).sum()))
p['age_clean'] = p['age'].where(p['age'].between(5, 120))
p['age_clean'].describe()

## 4. Check-in volume over time
Sparse, irregular per-user time series — motivates a temporal split and a
strictly-next-calendar-day target.

In [ ]:
by = p['date'].dt.to_period('M').value_counts().sort_index()
plt.figure(figsize=(8,3)); plt.plot([str(x) for x in by.index], by.values, color='#2563eb')
plt.xticks(range(0, len(by), 6), [str(by.index[i]) for i in range(0, len(by), 6)], rotation=45, ha='right')
plt.title('User-days per month'); plt.ylabel('user-days'); plt.tight_layout(); plt.show()

## 5. Build the next-day target + lagged history (H1 / H2 features)

In [ ]:
p = p.sort_values(['user_id', 'date'])
g = p.groupby('user_id')['sym_mean']
for k in (1, 2, 3):
    p[f'lag{k}_sym'] = g.shift(k)
# next calendar day's mean severity
nxt = p[['user_id', 'date', 'sym_mean']].copy()
nxt['date'] -= pd.Timedelta(days=1)
p = p.merge(nxt.rename(columns={'sym_mean': 'target'}), on=['user_id', 'date'], how='left')
data = p[p['target'].notna() & p['sym_mean'].notna()]
print('modeling rows:', len(data))
data[['sym_mean', 'lag1_sym', 'lag2_sym', 'lag3_sym', 'target']].corr()['target']

## 6. Leakage-safe temporal split + model vs naive baseline
Mirrors `src/make_features.py`. Preprocessing is fit on **train only** via a Pipeline.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_squared_error

FEATURES = ['sym_mean','lag1_sym','lag2_sym','lag3_sym','sym_count','cond_mean',
            'treat_count','food_count','tag_count','temperature_max','temperature_min',
            'humidity','pressure','precip_intensity','age_clean']
SPLIT = pd.Timestamp('2019-01-01')
tr, te = data[data['date'] < SPLIT], data[data['date'] >= SPLIT]

naive = np.sqrt(mean_squared_error(te['target'], te['sym_mean']))
pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', LinearRegression())])
cv = -cross_val_score(pipe, tr[FEATURES], tr['target'], cv=GroupKFold(5),
                      groups=tr['user_id'], scoring='neg_root_mean_squared_error')
pipe.fit(tr[FEATURES], tr['target'])
test_rmse = np.sqrt(mean_squared_error(te['target'], pipe.predict(te[FEATURES])))
print(f'naive baseline RMSE : {naive:.3f}')
print(f'grouped-CV RMSE     : {cv.mean():.3f} ± {cv.std():.3f}')
print(f'linreg test RMSE    : {test_rmse:.3f}  ({(naive-test_rmse)/naive*100:+.1f}% vs naive)')
print('H1 supported:', test_rmse < naive)

## 7. H2 — symptom history vs environment
Compare the predictive power of the two feature blocks under the same grouped CV.

In [ ]:
def block_rmse(cols):
    return -cross_val_score(pipe, tr[cols], tr['target'], cv=GroupKFold(5),
                            groups=tr['user_id'], scoring='neg_root_mean_squared_error').mean()
hist = block_rmse(['sym_mean','lag1_sym','lag2_sym','lag3_sym'])
env  = block_rmse(['temperature_max','temperature_min','humidity','pressure','precip_intensity'])
print(f'history-only CV RMSE     : {hist:.3f}')
print(f'environment-only CV RMSE : {env:.3f}')
print('H2 supported (history stronger):', hist < env)

---
**Takeaways:** H1 supported (model beats the naive baseline by ~12%); H2 supported
(a patient's recent symptom history predicts tomorrow far better than the weather).
Next: standardize `trackable_name`, add per-user baseline severity, try per-condition cohorts.